### Test if a pdf file needs to be OCR'ed in order to get the body text

Maybe this works?  Turns out I was wrong about a pdf being all image, so I moved on.

In [7]:
import pathlib as pl
refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of your .ipynb 
import sys
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
from icecream import ic

In [8]:
import fitz

def check_pdf_text_encoding(pdf_path):
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]
        
        # Try to extract text
        text = page.get_text()
        
        # Get images
        image_list = page.get_images()
        
        # Check page dimensions
        page_area = page.rect.width * page.rect.height
        
        # If there's no text but there are images covering most of the page
        # it likely means text is encoded as images
        if not text.strip() and image_list:
            for img in image_list:
                xref = img[0]
                image = doc.extract_image(xref)
                if image:
                    # Check if image covers most of the page
                    image_area = image["width"] * image["height"]
                    if image_area > (page_area * 0.5):
                        return True
    
    doc.close()
    return False

In [9]:
test_dir = refwrangle_dir / 'test'
pdf_path = test_dir / 'tmp_playwright.pdf'  # at the moment, I know that the text is image

ic(check_pdf_text_encoding(pdf_path))

ic| check_pdf_text_encoding(pdf_path): False


False